# SDG 3 Indicator Text Classification
**Group Assignment 2 — Complete Experiment Pipeline**

Run this notebook top-to-bottom in Google Colab. All dependencies install in Section 0.

In [ ]:
# Section 0: Install dependencies and configure paths
import subprocess, sys, os

# Install all requirements
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)

# NLTK downloads
import nltk
for pkg in ['punkt', 'stopwords', 'wordnet', 'punkt_tab', 'averaged_perceptron_tagger']:
    nltk.download(pkg, quiet=True)

# Make sure src/ and models/ are importable
if '.' not in sys.path:
    sys.path.insert(0, '.')

print('Setup complete.')


In [ ]:
# Download GloVe vectors (needed for Experiment 11)
# ~820MB — only downloads if not already present
import os, subprocess
if not os.path.exists('glove.6B.300d.txt'):
    print('Downloading GloVe 6B vectors (~820MB)...')
    subprocess.run(['wget', '-q', 'https://nlp.stanford.edu/data/glove.6B.zip'], check=True)
    subprocess.run(['unzip', '-q', 'glove.6B.zip', 'glove.6B.300d.txt'], check=True)
    os.remove('glove.6B.zip')
    print('GloVe ready.')
else:
    print('GloVe vectors already present.')


---
## Section 1 — Person A: EDA & Preprocessing
*Kayonga Elvis* — dataset understanding, text statistics, preprocessing pipeline.

Outputs: `data/devex_train_clean.csv`, `data/devex_test_clean.csv`, visualizations in `outputs/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

train_df = pd.read_csv('data/Devex_train.csv', encoding='latin-1', low_memory=False)
test_df  = pd.read_csv('data/Devex_test_questions.csv', encoding='latin-1', low_memory=False)

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
train_df.head(3)


In [ ]:
# Column detection
def detect_text_col(df):
    obj_cols = [c for c in df.columns if df[c].dtype == object]
    return max(obj_cols, key=lambda c: df[c].dropna().astype(str).str.len().mean())

def detect_id_col(df):
    # prefer columns whose name suggests an ID
    for c in df.columns:
        if any(kw in c.lower() for kw in ('id', 'key', 'uid', 'uuid')):
            return c
    # fallback: first column with all-unique values that isn't object type
    for c in df.columns:
        if df[c].dtype != object and df[c].nunique() == len(df):
            return c
    return df.columns[0]

def detect_label_cols(df, text_col, id_col):
    import re
    return [c for c in df.columns
            if c not in (text_col, id_col)
            and df[c].dropna().astype(str).str.contains(r'\d+\.\d+', regex=True).mean() > 0.3]

TEXT_COL   = detect_text_col(train_df)
ID_COL     = detect_id_col(train_df)
LABEL_COLS = detect_label_cols(train_df, TEXT_COL, ID_COL)

print(f'Text column : {TEXT_COL}')
print(f'ID column   : {ID_COL}')
print(f'Label cols  : {len(LABEL_COLS)} -> {LABEL_COLS[:3]} ...')


In [ ]:
# Discover unique labels
from sklearn.preprocessing import MultiLabelBinarizer

def build_label_lists(df, label_cols):
    rows = []
    for _, row in df[label_cols].iterrows():
        labels = [str(v).strip() for v in row if pd.notna(v) and str(v).strip() not in ('', 'NA', 'nan')]
        rows.append(labels)
    return rows

label_lists = build_label_lists(train_df, LABEL_COLS)
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(label_lists)
ALL_LABELS = list(mlb.classes_)

print(f'Unique labels: {len(ALL_LABELS)}')
print(f'Label matrix : {Y.shape}')


In [ ]:
# Label distribution
import os
os.makedirs('outputs', exist_ok=True)

label_counts = Y.sum(axis=0)
sorted_idx = np.argsort(label_counts)[::-1]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(ALL_LABELS)), label_counts[sorted_idx], color='steelblue')
ax.set_xticks(range(len(ALL_LABELS)))
ax.set_xticklabels([ALL_LABELS[i][:20] for i in sorted_idx], rotation=90, fontsize=7)
ax.set_title('Label Frequency Distribution (SDG 3 Indicators)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('outputs/label_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Imbalance ratio: {label_counts.max() / label_counts.min():.2f}')


In [ ]:
# Text statistics
doc_lengths = train_df[TEXT_COL].dropna().astype(str).str.split().str.len()
print(f'Avg tokens : {doc_lengths.mean():.1f}')
print(f'Median     : {doc_lengths.median():.0f}')
print(f'Min / Max  : {doc_lengths.min()} / {doc_lengths.max()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(doc_lengths, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Document Length (tokens)')
axes[0].set_xlabel('Tokens')
char_lengths = train_df[TEXT_COL].dropna().astype(str).str.len()
axes[1].hist(char_lengths, bins=50, color='coral', edgecolor='white')
axes[1].set_title('Document Length (characters)')
axes[1].set_xlabel('Characters')
plt.tight_layout()
plt.savefig('outputs/document_length_histogram.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Preprocessing pipeline (Person A)
import re
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

DOMAIN_ACRONYMS = {'sdg', 'who', 'hiv', 'tb', 'usaid', 'ngo', 'oda', 'oecd', 'wash', 'ncds'}
DOMAIN_ACRONYMS_UPPER = {a.upper() for a in DOMAIN_ACRONYMS}
STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
tokenizer  = RegexpTokenizer(r'\b[a-zA-Z]+\b')

def clean_html(text):
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'&[a-z]+;', ' ', text)
    return text

def preprocess_text(text):
    if pd.isna(text): return ''
    text = clean_html(str(text))
    text = text.lower()
    tokens = tokenizer.tokenize(text)
    cleaned = []
    for tok in tokens:
        if tok.upper() in DOMAIN_ACRONYMS_UPPER:
            cleaned.append(tok.upper())
            continue
        if tok in STOP_WORDS: continue
        lemma = lemmatizer.lemmatize(tok)
        if len(lemma) > 1:
            cleaned.append(lemma)
    return ' '.join(cleaned)

# Skip if clean CSVs already exist
if not os.path.exists('data/devex_train_clean.csv'):
    print('Preprocessing train...')
    train_df['clean_text'] = train_df[TEXT_COL].apply(preprocess_text)
    train_df.to_csv('data/devex_train_clean.csv', index=False)
    print('Preprocessing test...')
    test_df['clean_text'] = test_df[TEXT_COL].apply(preprocess_text)
    test_df.to_csv('data/devex_test_clean.csv', index=False)
    print('Clean CSVs saved.')
else:
    print('Clean CSVs already exist — skipping preprocessing.')
